# **Policy learning: from an effect estimate to an assignment rule**

Practice for the module [“Heterogeneous effects”](https://ai-interpretability.school/learn/5/0/0), lesson “How to read the estimate and where to go wrong”, step “From an estimate to a decision: policy learning”.

An estimate $\hat\tau(x)$ says how much the treatment changes an object's outcome. A decision answers a different question: whom to treat. In the lesson “How to read the estimate and where to go wrong” it was written as a rule $\pi(x)\in\{0,1\}$ and chosen by its value $V(\pi)$, the mean outcome that assignment by the rule would produce; the value is estimated with doubly robust effect scores $\hat\Gamma_i$ ([Athey and Wager, 2021](https://arxiv.org/abs/1702.02896)). Here we do this on a randomized e-mail experiment, a synthetic one modeled on the Hillstrom set, so that the true effect is known:

1. the data: who got an e-mail and who visited the site;
2. doubly robust effect scores $\hat\Gamma_i$ with cross-fitting;
3. the value of simple rules and the cost of an e-mail;
4. a policy tree `econml.policy.DRPolicyTree`: depth, value estimated on training and on held-out data;
5. comparison with the rule “treat where the CATE estimate exceeds the cost” and a rule with a budget;
6. where it breaks: the choice of the outcome the value is computed on, and the instability of a tree on small samples.

The notebook runs on CPU; the whole run takes two to three minutes.

In [ ]:
!pip install -q econml==0.17.0

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, train_test_split
from econml.policy import DRPolicyTree
from econml.dml import CausalForestDML

warnings.filterwarnings('ignore', message='.*encountered in matmul')   # spurious numpy warnings on some CPUs
SEED = 0
pd.set_option('display.width', 140)

## 1. The data: a synthetic e-mail campaign

The setting and the features are taken from the [MineThatData E-Mail Analytics Challenge](https://blog.minethatdata.com/2008/03/minethatdata-e-mail-analytics-and-data.html) (Kevin Hillstrom, 2008): customers of an online shop who bought within the last 12 months are randomly split into groups with and without an e-mail, and two weeks later it is recorded whether the person visited the site (`visit`), bought (`conversion`) and how much they spent (`spend`, dollars). The MineThatData set itself is published without a license, so the notebook generates an analog of it with the function `make_email_rct`: 42,000 rows, the women's e-mail ($T=1$) or no e-mail ($T=0$) with probability 1/2. The shares of the groups, visits, and purchases are close to the original set, and the true effect of the e-mail is known: in the synthetic data a formula sets it, and it lets us check whether a rule found the people the e-mail helps.

Features:

- `recency`, months since the last purchase;
- `history`, dollars spent over the year;
- `mens`, `womens`, whether the customer bought men's and women's merchandise;
- `newbie`, a new customer within the last 12 months;
- `zip_code`, the type of area (urban, suburban, rural), and `channel`, how they bought (phone, web, both).

The outcome for the rule is a site visit `visit`. Half of the data trains the rules, half is held out to estimate their value.

In [ ]:
def make_email_rct(n=42000, seed=SEED):
    rng = np.random.default_rng(seed)
    recency = rng.integers(1, 13, n)
    history = np.round(np.exp(rng.normal(5.0, 0.9, n)), 2)
    mens, womens = rng.binomial(1, 0.55, n), rng.binomial(1, 0.55, n)
    none, pick = (mens + womens) == 0, rng.binomial(1, 0.5, n)
    mens = np.where(none & (pick == 1), 1, mens)
    womens = np.where(none & (pick == 0), 1, womens)
    newbie = rng.binomial(1, 0.5, n)
    zip_code = rng.choice(['Urban', 'Suburban', 'Rural'], n, p=[0.4, 0.45, 0.15])
    channel = rng.choice(['Web', 'Phone', 'Multichannel'], n, p=[0.44, 0.44, 0.12])
    T = rng.binomial(1, 0.5, n)
    p0 = 1 / (1 + np.exp(-(-2.4 + 0.25 * womens + 0.15 * mens - 0.06 * (recency - 6)
                           + 0.25 * np.log(history / 150) + 0.3 * (channel == 'Web') - 0.2 * newbie)))
    tau = 0.072 * womens + 0.012 * (1 - womens) - 0.002 * (recency - 6)   # true effect of the e-mail on a visit
    visit = rng.binomial(1, np.clip(p0 + T * tau, 0, 1))
    conversion = visit * rng.binomial(1, 0.07, n)
    spend = np.round(conversion * np.exp(rng.normal(4.4, 0.6, n)), 2)
    df = pd.DataFrame(dict(recency=recency, history=history, mens=mens, womens=womens, zip_code=zip_code,
                           newbie=newbie, channel=channel, T=T, visit=visit, conversion=conversion, spend=spend))
    return df, tau


df, tau_true = make_email_rct()
T = df['T'].values
Xdf = pd.get_dummies(df[['recency', 'history', 'mens', 'womens', 'newbie', 'zip_code', 'channel']],
                     columns=['zip_code', 'channel']).astype(float)
X, names = Xdf.values, list(Xdf.columns)
Y = df.visit.values

tr, te = train_test_split(np.arange(len(df)), test_size=0.5, random_state=SEED, stratify=T)
print('customers in the two groups:', len(df), '| share who got an e-mail:', round(T.mean(), 3))
print('training:', len(tr), '| held-out estimate:', len(te))
print('share who visited the site: with e-mail', round(Y[T == 1].mean(), 4), '| without e-mail', round(Y[T == 0].mean(), 4))
print('true effect: mean', round(tau_true.mean(), 4), '| for buyers of the women\'s collection', round(tau_true[df.womens == 1].mean(), 4), '| for the rest', round(tau_true[df.womens == 0].mean(), 4))


## 2. Doubly robust effect scores

For every object we build a score of its effect

$$\hat\Gamma_i=\hat\mu_1(X_i)-\hat\mu_0(X_i)+\frac{T_i\,(Y_i-\hat\mu_1(X_i))}{\hat e(X_i)}-\frac{(1-T_i)\,(Y_i-\hat\mu_0(X_i))}{1-\hat e(X_i)},$$

where $\hat\mu_1$, $\hat\mu_0$ are the outcome models of the two groups and $\hat e$ the propensity model. This is the same pseudo-outcome as in the DR-learner. Each $\hat\Gamma_i$ alone is very noisy, but its mean over any group of objects is an unbiased estimate of the mean effect in that group if at least one of the two models is correct. The auxiliary models are trained on other folds (cross-fitting), and the scores of the training and the held-out half are computed separately: held-out scores must not depend on models that saw the training data.

**Task 1. Write a function `dr_scores(Xs, Ts, Ys, seed)` that, over five folds `KFold(5, shuffle=True, random_state=seed)`, trains on the other folds a `GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=SEED)` separately for groups $T=1$ and $T=0$ and a `LogisticRegression(max_iter=1000)` for the propensity, and returns $\hat\Gamma_i$ for the objects of the current fold. Compute `G_tr = dr_scores(X[tr], T[tr], Y[tr], 0)` and `G_te = dr_scores(X[te], T[te], Y[te], 1)` and compare the mean of `G_te` with the difference of mean outcomes on the held-out half.**

In [ ]:
# Your code here

## 3. The value of a rule and the cost of an e-mail

If an e-mail costs nothing and the effect is nonnegative for almost everyone, the best rule is to e-mail everyone, and there is nothing to choose. Let us make the problem realistic for a business: suppose an e-mail costs $c$ visits; this can be the sending cost, the risk of an unsubscribe, or mailing fatigue, expressed in the same units as the outcome. Then the gain of a rule relative to “e-mail nobody” is

$$\hat V_c(\pi)=\frac1n\sum_i\pi(X_i)\,(\hat\Gamma_i-c).$$

We take $c=0.045$, roughly the mean effect of the e-mail on visits: at this cost “e-mail everyone” gains almost nothing, and only a rule that tells apart those whom the e-mail helps gains anything.

**Task 2. Write a function `policy_value(pi, G, c=COST)` and compute on the held-out half the gain of three rules: “e-mail everyone”, “e-mail those who bought women's merchandise” (`womens == 1`) and “e-mail those who did not buy men's merchandise” (`mens == 0`). For each print the share e-mailed and the gain.**

In [ ]:
COST = 0.045
col = lambda name, rows: X[rows][:, names.index(name)]

In [ ]:
# Your code here

## 4. The policy tree

`econml.policy.DRPolicyTree` builds $\hat\Gamma_i$ on the training data itself and searches for a tree of given depth that maximizes $\frac1n\sum_i\pi(X_i)\hat\Gamma_i$. We pass the cost of the e-mail through the outcome: subtracting $c$ from the outcome of everyone e-mailed lowers each effect by exactly $c$.

We train trees of depth 1, 2, 3, and 4 (`min_samples_leaf=500`) and for each compare the gain computed on the training scores `G_tr` with the gain on the held-out `G_te`. The tree searches for the rule with its own internal scores $\hat\Gamma_i$, while `G_tr` come from our models of task 1, so the gain on `G_tr` is also a check, although on the same objects.

In [ ]:
def policy_tree(depth, rows=tr, outcome=Y, c=COST, leaf=500):
    return DRPolicyTree(model_regression=GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=SEED),
                        model_propensity=LogisticRegression(max_iter=1000), max_depth=depth, min_samples_leaf=leaf,
                        cv=5, random_state=SEED).fit(outcome[rows] - c * T[rows], T[rows], X=X[rows])

trees = {d: policy_tree(d) for d in (1, 2, 3, 4)}
depth_tab = pd.DataFrame({d: {'share e-mailed': t.predict(X[te]).mean(),
                              'gain on training G_tr': policy_value(t.predict(X[tr]), G_tr),
                              'gain on held-out G_te': policy_value(t.predict(X[te]), G_te)}
                          for d, t in trees.items()}).T
depth_tab.index.name = 'depth'
print(depth_tab.round(5))

Depths 1, 2, and 3 give the same rule: e-mail those who bought women's merchandise. The synthetic data show that this is the best rule: the true effect of the e-mail is 0.071 for buyers of the women's collection and 0.011 for the rest (output of section 1), and the e-mail costs 0.045. Depth 4 adds splits, and the held-out gain decreases: deeper is not better, the extra splits catch noise in the scores $\hat\Gamma_i$.

The whole depth-2 tree: each node shows a feature and a threshold, each leaf the recommended action (0, do not e-mail; 1, e-mail).

In [ ]:
plt.figure(figsize=(12, 5))
trees[2].plot(feature_names=names, treatment_names=['no e-mail', 'e-mail'], fontsize=9)
plt.show()

## 5. A rule from the CATE estimate and a rule with a budget

Another path to a rule is to estimate the CATE first and then treat where $\hat\tau(x)>c$. We estimate the CATE with the causal forest `CausalForestDML` on the training half and compare the held-out gain of this rule with the depth-2 tree. The tree can be read in full; the CATE rule cannot: to explain why a particular person got an e-mail, one would have to explain the forest itself.

The budget. The lesson says constraints are part of the problem. Suppose there are e-mails for only 20% of customers. A budget rule from the CATE estimate sends the e-mail to the 20% of customers with the largest $\hat\tau(x)$.

In [ ]:
cfp = CausalForestDML(model_y=GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=SEED),
                      model_t=GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=SEED),
                      discrete_treatment=True, n_estimators=500, min_samples_leaf=50, cv=5, random_state=SEED)
cfp.fit(Y[tr], T[tr], X=X[tr])
tau_te = cfp.effect(X[te])
pi_cate = (tau_te > COST).astype(float)
pi_tree = trees[2].predict(X[te]).astype(float)
print(f'rule “CATE estimate > c”: share {pi_cate.mean():.3f}, gain {policy_value(pi_cate, G_te):+.5f}')
print(f'depth-2 tree:             share {pi_tree.mean():.3f}, gain {policy_value(pi_tree, G_te):+.5f}')
print(f'share of customers on which the two rules agree: {(pi_cate == pi_tree).mean():.3f}')

**Task 3. A rule with a 20% budget: e-mail the 20% of customers of the held-out half with the largest CATE estimate `tau_te` and compute the increase in the share of site visitors per customer of the whole held-out half, $\frac1n\sum_i\pi(X_i)\hat\Gamma_i$ (without the e-mail cost: the budget already limits the number of e-mails). Compare with a random 20% (mean over 200 random draws with the generator `np.random.default_rng(SEED)`) and draw the curve: share e-mailed from 0 to 1 on the X axis, the increase on the Y axis, for the order by `tau_te` and for a random order.**

In [ ]:
# Your code here

## 6. Where it breaks

### 6.1. The value is computed for the chosen outcome

The lesson calls this the fourth trap: a rule is best only for the outcome its value was computed on. We train a depth-2 tree on another outcome, purchase amount `spend` in dollars, with an e-mail cost equal to the mean effect of the e-mail on `spend` in the training half, and compare whom the two rules e-mail and what gain each gives on both outcomes.

In [ ]:
S = df.spend.values
cost_spend = S[tr][T[tr] == 1].mean() - S[tr][T[tr] == 0].mean()
tree_spend = policy_tree(2, outcome=S, c=cost_spend)
pi_spend = tree_spend.predict(X[te]).astype(float)
G_te_spend = dr_scores(X[te], T[te], S[te], 1)
print(f'e-mail cost for spend: {cost_spend:.3f} dollars')
print(f'tree by visit: share {pi_tree.mean():.3f} | tree by spend: share {pi_spend.mean():.3f}')
print(f'share of customers on which the two trees agree: {(pi_tree == pi_spend).mean():.3f}')
table = pd.DataFrame({'gain in visit': [policy_value(p, G_te) for p in (pi_tree, pi_spend)],
                      'gain in spend, dollars': [policy_value(p, G_te_spend, cost_spend) for p in (pi_tree, pi_spend)]},
                     index=['tree by visit', 'tree by spend'])
print(table.round(5))
plt.figure(figsize=(12, 5))
tree_spend.plot(feature_names=names, treatment_names=['no e-mail', 'e-mail'], fontsize=9)
plt.show()

The two trees agree on only 29% of the customers: the tree by `spend` e-mails 5.7% of the customers, the tree by `visit` 65.3%. Each tree is better on its own outcome: on `spend` the tree by `spend` gives 0.01537 dollars per customer against 0.00049 for the tree by `visit`, and on visits it goes negative, −0.00383. About 1% of customers make a purchase, the scores $\hat\Gamma_i$ for `spend` are much noisier, and the rule based on them rests on a small group. Choosing the outcome is both choosing the goal and choosing how noisy a value one will have to estimate.

### 6.2. Instability of the tree on a small sample

The guarantee of Athey and Wager is about regret: the value of the found rule falls short of the best rule of the class by an amount of order $1/\sqrt n$. On a small sample this shortfall is large, and the tree changes from subsample to subsample.

**Task 4. For subsample sizes of 500, 2000, and 8000 objects of the training half and seeds 0–9, train a depth-2 tree (`policy_tree(2, rows=...)`, `leaf=50`; take the subsample as `np.random.default_rng(seed).choice(tr, size, replace=False)`). For each size print the mean and the standard deviation of the held-out gain and the mean share of held-out customers on which the tree agrees with `trees[2]`, trained on the whole training half.**

In [ ]:
# Your code here

## Implementations

- `econml.policy.DRPolicyTree` and `econml.policy.DRPolicyForest`: a policy tree and a policy forest on doubly robust scores; they do not take a cost or a budget directly, the cost is passed through the outcome, as here;
- The R package [policytree](https://grf-labs.github.io/policytree/): `double_robust_scores()` builds $\hat\Gamma_i$ from a grf causal forest, `policy_tree()` finds the exactly optimal tree of given depth by exhaustive search.

## What to take away

- **A rule is chosen by its value, not by the accuracy of the effect estimate.** Doubly robust scores $\hat\Gamma_i$ are noisy one by one, but their mean over a group estimates the effect in the group, and that is enough to compare rules.
- **Without a cost of treatment there is no problem.** With a free e-mail and a nonnegative effect, the best rule is to e-mail everyone.
- **The value of a rule is estimated on held-out data, and deeper is not better.** Here a single split on `womens` decides everything; the depth-4 tree gives a smaller held-out gain.
- **The rule depends on the outcome.** Trees by visits and by purchases agree on less than a third of the customers, and each is better on its own outcome and worse on the other's. The goal and the cost must be written down before searching for the rule, and a rule based on a noisy outcome (rare purchases) rests on a small group.
- **A small sample means an unstable rule.** Regret decays as $1/\sqrt n$, and with hundreds of objects the tree changes from subsample to subsample.

Sources:
1. [Athey and Wager, 2021. Policy learning with observational data](https://arxiv.org/abs/1702.02896).
2. [Hillstrom, 2008. The MineThatData E-Mail Analytics and Data Mining Challenge](https://blog.minethatdata.com/2008/03/minethatdata-e-mail-analytics-and-data.html), the model for the setting and the features of the synthetic data.
3. [EconML](https://github.com/py-why/EconML), [policytree](https://grf-labs.github.io/policytree/).